# Project 4: Data Type Memory & Performance Benchmark Lab

**Objective:** see how the dtype (int8 to int64, float16 to float64) changes memory, speed and precision.

In [ ]:
import numpy as np
import timeit
import matplotlib.pyplot as plt

## Step 1: Make the same array in every dtype

In [ ]:
# 1 million numbers from 0 to 99 (0-99 fits in int8)
base = np.arange(1000000) % 100

names = ["int8", "int16", "int32", "int64", "float16", "float32", "float64"]

arrays = {}
for n in names:
    arrays[n] = base.astype(n)

for n in names:
    print(n, arrays[n][:5])

## Step 2: size, itemsize and nbytes

In [ ]:
print("dtype    size    itemsize    nbytes")
mem = []
for n in names:
    a = arrays[n]
    print(n, a.size, a.itemsize, a.nbytes)
    mem.append(a.nbytes / 1024 / 1024)   # in MB

itemsize is bytes per number. int8 = 1 byte, int64 = 8 bytes, so int8 uses 8x less memory.

## Step 3: Time sum and multiply

In [ ]:
sum_time = []
mul_time = []

for n in names:
    arr = arrays[n]
    t1 = timeit.timeit("arr.sum()", globals=globals(), number=100) / 100 * 1000
    t2 = timeit.timeit("arr * 2", globals=globals(), number=100) / 100 * 1000
    sum_time.append(t1)
    mul_time.append(t2)
    print(n, "sum:", round(t1, 3), "ms   multiply:", round(t2, 3), "ms")

Smaller types are faster for multiply. float16 is slow because the CPU has no fast support for it.

In [ ]:
for n in names:
    print(n, arrays[n].sum())

float16 sum gives inf because its max value is 65504 and the real sum is about 49 million.

## Step 4: Overflow and precision loss

In [ ]:
print(np.iinfo(np.int8))

x = np.array([100, 127, 128, 200, 300])
print(x.astype(np.int8))
print(x.astype(np.int16))

a = np.array([100], dtype=np.int8)
print(a + a)

int8 can only hold -128 to 127, so 200 wraps around to -56. There is no error, so it is easy to miss.

In [ ]:
print(np.finfo(np.float16))
print(np.finfo(np.float32))
print(np.finfo(np.float64))

x = np.float64(1234.56789123456789)
print(x)
print(x.astype(np.float32))
print(x.astype(np.float16))

print(np.float16(0.1) + np.float16(0.2))
print(np.float64(0.1) + np.float64(0.2))

print(np.float64(70000).astype(np.float16))

float16 keeps about 3 digits, float32 about 7 and float64 about 15. float16 cannot store anything above 65504 (gives inf).

## Step 5: Memory vs time charts

In [ ]:
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.bar(names, mem)
plt.title("Memory (MB)")
plt.xticks(rotation=45)

plt.subplot(1, 3, 2)
plt.bar(names, sum_time, color="orange")
plt.title("Sum time (ms)")
plt.xticks(rotation=45)

plt.subplot(1, 3, 3)
plt.bar(names, mul_time, color="green")
plt.title("Multiply time (ms)")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()